<a href="https://colab.research.google.com/github/chamarairesh1982/LearnPython/blob/main/chamara_Iris_KNN_DecisionTree_SVM_Teaching_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning Classification with Iris Dataset

## K-Nearest Neighbors (KNN), Decision Tree, and Support Vector Machine (SVM)
We will use the classic **Iris dataset** to understand a complete machine-learning workflow:

1. Load and inspect the dataset
2. Prepare the data
3. Split data into training and testing sets
4. Train and evaluate KNN
5. Train and evaluate a Decision Tree
6. Train and evaluate an SVM
7. Compare the three algorithms
8. Understand important hyperparameters

**Learning goal:** Understand not only how to run each algorithm, but also *why* preprocessing and hyperparameter choices matter.

## 1. Import Required Libraries

We will use **pandas** for tabular data handling, **NumPy** for numerical operations, **matplotlib/seaborn** for visualization, and **scikit-learn** for machine-learning algorithms and evaluation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print('Libraries imported successfully!')

## 2. Load the Iris Dataset

The Iris dataset contains measurements of iris flowers from three species:

- **Setosa**
- **Versicolor**
- **Virginica**

There are four input features:

- Sepal length
- Sepal width
- Petal length
- Petal width

The target variable is the flower species.

In [ ]:
iris = load_iris()

X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name='target')

# Human-readable class names
class_names = iris.target_names

print('Feature names:')
print(iris.feature_names)

print('\nTarget classes:')
print(class_names)

print('\nDataset shape:', X.shape)

## 3. Inspect the Dataset

Before training a model, we should understand the data.

Important checks include:

- Number of rows and columns
- Data types
- Missing values
- Duplicate rows
- Distribution of target classes

This is part of **exploratory data analysis (EDA)**.

In [ ]:
df = X.copy()
df['species'] = y.map(dict(enumerate(class_names)))

display(df.head())

print('Dataset information:')
df.info()

print('\nMissing values:')
print(df.isnull().sum())

print('\nDuplicate rows:', df.duplicated().sum())

print('\nClass distribution:')
print(df['species'].value_counts())

### Observation

The Iris dataset is particularly convenient for teaching because it is small, clean, and balanced. It has no missing values in the original dataset.

In a real-world project, missing values, invalid values, outliers, duplicate records, and class imbalance may require additional preprocessing.

## 4. Visualize the Features

Visualization helps us understand whether the classes are naturally separated.

A pair plot is useful here because it lets us see relationships between pairs of features.

In [ ]:
sns.pairplot(df, hue='species')
plt.show()

## 5. Prepare the Data

We separate the dataset into:

- **X** → input features
- **y** → target/output class

Then we split the data into training and testing sets.

### Why do we split the data?

The model should learn from the **training set** and be evaluated on unseen data called the **test set**.

Here we use 80% for training and 20% for testing.

### Why `stratify=y`?

It keeps approximately the same proportion of each flower species in both training and testing sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print('Training samples:', len(X_train))
print('Testing samples :', len(X_test))

print('\nTraining class distribution:')
print(y_train.value_counts().sort_index())

print('\nTesting class distribution:')
print(y_test.value_counts().sort_index())

## 6. Feature Scaling

**Feature scaling** transforms numerical features to a comparable scale.

A common method is standardization:

$$
z = \frac{x - \mu}{\sigma}
$$

where:

- $x$ = original value
- $\mu$ = training-set mean
- $\sigma$ = training-set standard deviation

### Why is scaling important?

KNN uses distances between data points, and SVM is sensitive to the scale of features. Therefore, scaling is generally important for both algorithms.

Decision Trees are different: they split data based on feature thresholds, so feature scaling is usually **not necessary**.

### Important: avoid data leakage

The scaler must be **fit only on the training data**. A scikit-learn `Pipeline` makes this easy and safe.

# Part A — K-Nearest Neighbors (KNN)

## 7. What is KNN?

K-Nearest Neighbors is a **distance-based** supervised learning algorithm.

For a new data point, KNN:

1. Calculates its distance from training examples.
2. Finds the **K nearest** examples.
3. Looks at their classes.
4. Assigns the class with the majority vote.

For example, if `K = 5` and 4 of the 5 nearest flowers are Virginica, the new flower is predicted as Virginica.

## 8. Important KNN Hyperparameters

### `n_neighbors`

Number of neighbors used for voting.

- Small K → flexible model, but can be sensitive to noise.
- Large K → smoother model, but may miss local patterns.

### `weights`

- `uniform` → every neighbor has equal importance.
- `distance` → closer neighbors have more influence.

### `metric`

Distance measure used by KNN. Common choices include:

- `euclidean`
- `manhattan`

For teaching, start with the default Euclidean distance.

In [ ]:
# KNN with scaling
knn_model = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(
        n_neighbors=5,
        metric='euclidean'
    ))
])

knn_model.fit(X_train, y_train)
knn_predictions = knn_model.predict(X_test)

knn_accuracy = accuracy_score(y_test, knn_predictions)

print(f'KNN Accuracy: {knn_accuracy:.4f} ({knn_accuracy * 100:.2f}%)')

## 9. KNN Evaluation

Accuracy is:

$$
\text{Accuracy} = \frac{\text{Correct Predictions}}{\text{Total Predictions}}
$$

We can also inspect the confusion matrix and classification report.

In [ ]:
print('KNN Classification Report:')
print(classification_report(y_test, knn_predictions, target_names=class_names))

cm_knn = confusion_matrix(y_test, knn_predictions)

sns.heatmap(
    cm_knn,
    annot=True,
    fmt='d',
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('KNN Confusion Matrix')
plt.show()

# Part B — Decision Tree

## 10. What is a Decision Tree?

A Decision Tree makes predictions by repeatedly asking questions about feature values.

For example:

`Is petal length <= some threshold?` → go left/right → ask another question → eventually reach a leaf.

The final leaf represents the predicted class.

Decision Trees are easy to visualize and explain, which makes them excellent for teaching.

## 11. Important Decision Tree Hyperparameters

### `max_depth`

Maximum depth of the tree.

- Small value → simpler tree, potentially underfitting.
- Large value → more complex tree, potentially overfitting.

### `min_samples_split`

Minimum number of samples required to split an internal node.

Increasing it generally makes the tree simpler.

### `min_samples_leaf`

Minimum number of samples allowed in a leaf node.

Larger values can reduce overfitting.

### `criterion`

Measures how good a split is. Common choices include:

- `gini`
- `entropy`

### `random_state`

Controls randomness so results can be reproduced.

In [ ]:
# Decision Tree does not require feature scaling
tree_model = DecisionTreeClassifier(
    criterion='gini',
    max_depth=3,
    random_state=42
)

tree_model.fit(X_train, y_train)
tree_predictions = tree_model.predict(X_test)

tree_accuracy = accuracy_score(y_test, tree_predictions)

print(f'Decision Tree Accuracy: {tree_accuracy:.4f} ({tree_accuracy * 100:.2f}%)')

## 12. Visualize the Decision Tree

This visualization is useful when explaining how a tree makes decisions.

Notice how the tree selects a feature and threshold at each internal node.

In [ ]:
plt.figure(figsize=(18, 10))

plot_tree(
    tree_model,
    feature_names=iris.feature_names,
    class_names=class_names,
    filled=True,
    rounded=True,
    fontsize=10
)

plt.title('Decision Tree for Iris Classification')
plt.show()

In [ ]:
print('Decision Tree Classification Report:')
print(classification_report(y_test, tree_predictions, target_names=class_names))

cm_tree = confusion_matrix(y_test, tree_predictions)

sns.heatmap(
    cm_tree,
    annot=True,
    fmt='d',
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Decision Tree Confusion Matrix')
plt.show()

# Part C — Support Vector Machine (SVM)

## 13. What is SVM?

Support Vector Machine tries to find a decision boundary that separates classes while maximizing the **margin** between the classes.

The training examples closest to the boundary are called **support vectors**.

SVM can also use kernels to create non-linear decision boundaries.

## 14. Important SVM Hyperparameters

### `C`

Controls the trade-off between a wide margin and classification errors.

- Small `C` → allows more training errors and a smoother boundary.
- Large `C` → penalizes errors strongly and can create a more complex boundary.

### `kernel`

Defines the mathematical transformation used to separate the data.

Common options:

- `linear`
- `rbf`
- `poly`

### `gamma`

Important for RBF, polynomial, and sigmoid kernels.

- Low gamma → each point has a broader influence.
- High gamma → each point has a more local influence and can make the model very flexible.

### `degree`

Controls the polynomial degree when `kernel='poly'`.

In [ ]:
# SVM with scaling
svm_model = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', C=1.0, gamma='scale'))
])
svm_model.fit(X_train, y_train)
svm_predictions = svm_model.predict(X_test)

svm_accuracy = accuracy_score(y_test, svm_predictions)

print(f'SVM Accuracy: {svm_accuracy:.4f} ({svm_accuracy * 100:.2f}%)')

In [ ]:
print('SVM Classification Report:')
print(classification_report(y_test, svm_predictions, target_names=class_names))

cm_svm = confusion_matrix(y_test, svm_predictions)

sns.heatmap(
    cm_svm,
    annot=True,
    fmt='d',
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('SVM Confusion Matrix')
plt.show()

# 15. Compare the Three Algorithms

Let's place the test-set accuracy of all three models side by side.

**Important teaching point:** The exact accuracy can change when the train/test split or hyperparameters change. A single train/test split should therefore not be treated as proof that one algorithm is universally better.

In [ ]:
results = pd.DataFrame({
    'Algorithm': ['KNN', 'Decision Tree', 'SVM'],
    'Accuracy': [knn_accuracy, tree_accuracy, svm_accuracy]
})

results['Accuracy (%)'] = results['Accuracy'] * 100
display(results.sort_values('Accuracy', ascending=False).reset_index(drop=True))

In [ ]:
plt.figure(figsize=(8, 5))
bars = plt.bar(results['Algorithm'], results['Accuracy (%)'])
plt.ylim(0, 100)
plt.ylabel('Accuracy (%)')
plt.xlabel('Algorithm')
plt.title('Iris Classification Accuracy Comparison')

for bar, value in zip(bars, results['Accuracy (%)']):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 1,
        f'{value:.2f}%',
        ha='center'
    )

plt.show()

# 16. Hyperparameter Experiment — KNN

One of the easiest ways to understand hyperparameters is to change them and observe the result.

Here we test several values of `n_neighbors`.

**Question for students:** Why might accuracy increase and then decrease as K changes?

In [ ]:
k_values = range(1, 16)
knn_accuracies = []

for k in k_values:
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=k))
    ])
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    knn_accuracies.append(accuracy_score(y_test, pred))

plt.figure(figsize=(8, 5))
plt.plot(list(k_values), np.array(knn_accuracies) * 100, marker='o')
plt.xlabel('Number of Neighbors (K)')
plt.ylabel('Test Accuracy (%)')
plt.title('KNN: Effect of K on Test Accuracy')
plt.xticks(list(k_values))
plt.grid(True)
plt.show()

# 17. Hyperparameter Experiment — Decision Tree

Let's change `max_depth` and observe how tree complexity affects accuracy.

This demonstrates the idea of **underfitting vs. overfitting**:

- Tree too simple → may underfit.
- Tree too complex → may overfit.
- A suitable complexity → better generalization.

In [ ]:
depth_values = range(1, 11)
tree_accuracies = []

for depth in depth_values:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    tree_accuracies.append(accuracy_score(y_test, pred))

plt.figure(figsize=(8, 5))
plt.plot(list(depth_values), np.array(tree_accuracies) * 100, marker='o')
plt.xlabel('Maximum Tree Depth')
plt.ylabel('Test Accuracy (%)')
plt.title('Decision Tree: Effect of max_depth')
plt.xticks(list(depth_values))
plt.grid(True)
plt.show()

# 18. Hyperparameter Experiment — SVM

For SVM, `C` is one of the most important hyperparameters.

We will compare several values while keeping the RBF kernel.

In [ ]:
c_values = [0.01, 0.1, 1, 10, 100]
svm_accuracies = []

for c in c_values:
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(kernel='rbf', C=c, gamma='scale'))
    ])
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    svm_accuracies.append(accuracy_score(y_test, pred))

plt.figure(figsize=(8, 5))
plt.semilogx(c_values, np.array(svm_accuracies) * 100, marker='o')
plt.xlabel('C')
plt.ylabel('Test Accuracy (%)')
plt.title('SVM: Effect of C')
plt.grid(True)
plt.show()

# 19. Cross-Validation — A Better Evaluation Method

A single train/test split can give a lucky or unlucky result. **Cross-validation** provides a more reliable estimate.

In k-fold cross-validation:

1. The training data is divided into `k` folds.
2. The model trains on `k-1` folds.
3. It validates on the remaining fold.
4. This repeats until every fold has been used for validation.
5. The scores are averaged.

Here we use **5-fold cross-validation**.

In [ ]:
from sklearn.model_selection import cross_val_score

models = {
    'KNN': knn_model,
    'Decision Tree': tree_model,
    'SVM': svm_model
}

cv_results = []

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    cv_results.append({
        'Algorithm': name,
        'Mean CV Accuracy': scores.mean(),
        'Std Dev': scores.std()
    })
    print(f'{name}:')
    print('  Fold accuracies:', np.round(scores, 4))
    print(f'  Mean accuracy : {scores.mean():.4f}')
    print(f'  Std deviation : {scores.std():.4f}\n')

cv_results_df = pd.DataFrame(cv_results)
display(cv_results_df)

# 20. Quick Hyperparameter Reference

| Algorithm | Important Hyperparameters | Main Effect |
|---|---|---|
| **KNN** | `n_neighbors` | Controls how many neighbors vote |
| **KNN** | `weights` | Equal vs. distance-based voting |
| **KNN** | `metric` | Distance calculation |
| **Decision Tree** | `max_depth` | Controls maximum tree complexity |
| **Decision Tree** | `min_samples_split` | Minimum samples needed to split |
| **Decision Tree** | `min_samples_leaf` | Minimum samples in a leaf |
| **Decision Tree** | `criterion` | Measures split quality |
| **SVM** | `C` | Controls penalty for classification errors |
| **SVM** | `kernel` | Shape/type of decision boundary |
| **SVM** | `gamma` | Controls influence of individual points for non-linear kernels |
| **SVM** | `degree` | Polynomial kernel degree |

### Scaling reminder

- **KNN:** scaling is important.
- **SVM:** scaling is generally important.
- **Decision Tree:** scaling is normally unnecessary.

# 21. Key Takeaways for Students

### KNN

- Simple and intuitive.
- Makes predictions using nearby training examples.
- Sensitive to feature scale.
- Prediction can become expensive for very large datasets.

### Decision Tree

- Easy to understand and visualize.
- Does not normally require feature scaling.
- Can overfit if allowed to become too complex.
- `max_depth`, `min_samples_split`, and `min_samples_leaf` are useful controls for complexity.

### SVM

- Finds a decision boundary with a large margin.
- Can model non-linear boundaries using kernels.
- Usually benefits from feature scaling.
- `C`, `kernel`, and `gamma` are important hyperparameters.

### Most important general lesson

**Machine learning is not just about choosing an algorithm.** A good workflow includes data inspection, preprocessing, correct train/test separation, model training, evaluation, hyperparameter tuning, and validation.

# 22. Student Exercises

Try these exercises after completing the notebook:

### Exercise 1 — KNN
Change `n_neighbors` from 1 to 20. Which value gives the best test accuracy?

### Exercise 2 — KNN weights
Compare `weights='uniform'` with `weights='distance'`. What changes?

### Exercise 3 — Decision Tree
Try `max_depth=1`, `2`, `3`, `5`, and `None`. How does the tree change?

### Exercise 4 — Decision Tree overfitting
Compare training accuracy and test accuracy for a very deep tree. Is there evidence of overfitting?

### Exercise 5 — SVM kernels
Compare `linear`, `rbf`, and `poly` kernels. Which performs best on this split?

### Exercise 6 — SVM C
Try `C=0.01`, `0.1`, `1`, `10`, and `100`. Explain why changing C can change the decision boundary.

### Exercise 7 — Scaling experiment
Train KNN and SVM without scaling. Compare the result with the scaled versions. Explain why scaling matters.